In [1]:
from reasonable_crowd.parse_map import parse_map
from reasonable_crowd.dataset import build_evaluation_dataset, get_trajectories, load_annotations
import os
from rulebook_benchmark.realization import VariableHandler
import random
import shapely
from rulebook_benchmark.rule_functions import RuleEngine
import networkx as nx
from rulebook_benchmark.rule_functions import (
    f1, f2, f3, f4, f5, f6, f7, f8, f9, f11, f12, f13, f15, f17, f18, f11_v, f12_v, f13_v, f7_alt
)

from rulebook_benchmark.rulebook import Rulebook
from reasonable_crowd.InPlaceRulebook import InPlaceRulebook
from rulebook_benchmark.rulebook import Relation
import numpy as np
import pandas as pd
from reasonable_crowd.optimization import cache_rule_evaluations, optimize_rulebook_grid_bruteforce_with_validation, simulated_annealing, simulated_annealing_with_validation, number_of_unique_rulebooks, find_scenario_rulebooks, shuffle_rulebook, find_scenario_groups, brute_force_group_optimization, greedy_group_optimization, greedy_group_optimization_with_validation, group_nodes_by_level, brute_force_group_optimization_with_validation, group_rulebook
import pickle
from sklearn.model_selection import train_test_split
from reasonable_crowd.evaluation import evaluate_rulebook_with_cache, evaluate_rulebook, get_rulebook_results, evaluate_rulebook_with_result_dict
from sklearn.model_selection import KFold
from reasonable_crowd.visualization import plot_topological_graph, plot_two_rulebooks_side_by_side, plot_group_topological_graph
from rulebook_benchmark.plotting import compare_realizations_gif, animate_realization
from IPython.display import HTML

path_to_reasonable_crowd = "../../../Reasonable-Crowd"
map_directory = path_to_reasonable_crowd + '/maps'
trajectory_directory = path_to_reasonable_crowd + '/trajectories'
rule_id_to_name = {1: "vru_collision", 2: "vehicle_collision", 3: "drivable_area", 4: "vru_ttc", 5: "vru_acknowledgement", 6: "vehicle_ttc", 7:"correct_side", 8: "vru_offroad", 9: "vru_onroad", 11: "front_clearance", 12: "left_clearance", 13: "right_clearance", 15: "speed_limit", 17: "lane_keeping", 18: "lane_centering"}
network_U = parse_map(map_directory, 'U')
network_S = parse_map(map_directory, 'S')

output_directory = 'outputs'
output_file = os.path.join(output_directory, 'results_scenic.txt')

trajectories = get_trajectories(output_directory, trajectory_directory, network_U, network_S)

trajectories_dict = {}
for filename, realization in trajectories:
    trajectories_dict[filename[:-5]] = realization  # remove .json extension

data = load_annotations(path_to_reasonable_crowd)

X, y, y_votes, y_agreement = build_evaluation_dataset(data)
# create pandas dataframe
df = pd.DataFrame(columns=['X', 'y', 'votes', 'agreement'])
df['X'] = X
df['y'] = y
df['votes'] = y_votes
df['agreement'] = y_agreement
print(df.head())



# Prepare data
X = df['X'].tolist()
y = df['y'].tolist()
votes = df['votes'].tolist()




/Users/ekin/Scenic/src/scenic/core/errors.py:271: UserWarning: unable to install sys.excepthook to format Scenic backtraces
  warnings.warn("unable to install sys.excepthook to format Scenic backtraces")


Loading cached trajectories...
                X                y    votes  agreement
0  (U_1-a, U_1-b)  Relation.LARGER  (14, 0)   1.000000
1  (U_1-a, U_1-c)  Relation.LARGER  (14, 0)   1.000000
2  (U_1-a, U_1-d)  Relation.LARGER  (12, 2)   0.714286
3  (U_1-a, U_1-e)  Relation.LARGER  (10, 4)   0.428571
4  (U_1-a, U_1-f)  Relation.LARGER  (14, 0)   1.000000


In [2]:
rb = Rulebook(rule_file="reasonable_crowd_rule_functions.py", rulebook_file="reasonable_crowd_5.graph")
rule_id_to_rule = {1: f1, 2: f2, 3: f3, 4: f4, 5: f5, 6: f6, 7: f7, 8: f8, 9: f9, 11: f11, 12: f12, 13: f13, 15: f15, 17: f17, 18: f18}
rule_id_to_rule_v = {1: f1, 2: f2, 3: f3, 4: f4, 5: f5, 6: f6, 7: f7, 8: f8, 9: f9, 11: f11_v, 12: f12_v, 13: f13_v, 15: f15, 17: f17, 18: f18}
rule_id_to_rule_correct_side_alt = {1: f1, 2: f2, 3: f3, 4: f4, 5: f5, 6: f6, 7: f7_alt, 8: f8, 9: f9, 11: f11, 12: f12, 13: f13, 15: f15, 17: f17, 18: f18}


rulebook = InPlaceRulebook(rb.priority_graph, rule_id_to_rule)

rb_v = InPlaceRulebook(rb.priority_graph, rule_id_to_rule_v)
rb_correct_side_alt = InPlaceRulebook(rb.priority_graph, rule_id_to_rule_correct_side_alt)


In [3]:
groups = [[1, 2], [3, 7], [8, 9, 11, 12, 13], [17, 18, 15], [4, 5, 6]]
name_to_group = {"safety-critical": groups[0], "operation-limit": groups[1], "safety-enhancing": groups[2], "predictability": groups[3], "precautionary": groups[4]}
group_to_name = {tuple(value): key for key, value in name_to_group.items()}
default_groups = group_nodes_by_level(rulebook.in_place_priority_graph)
rulebook = group_rulebook(rulebook, groups, keep_relations=True)
rb_v = group_rulebook(rb_v, groups, keep_relations=True)
rb_correct_side_alt = group_rulebook(rb_correct_side_alt, groups, keep_relations=True)

In [5]:
base_results = get_rulebook_results(rulebook, trajectories_dict)
base_correct, base_equal, base_incomparable, base_total, base_accuracy, base_weighted_accuracy, base_reasons, base_predictions = evaluate_rulebook_with_result_dict(rulebook, base_results, X, y, votes)
print(base_accuracy)

TypeError: '<' not supported between instances of 'Result' and 'Result'

In [ ]:
v_results = get_rulebook_results(rb_v, trajectories_dict)
v_correct, v_equal, v_incomparable, v_total, v_accuracy, v_weighted_accuracy, v_reasons, v_predictions = evaluate_rulebook_with_result_dict(rb_v, v_results, X, y, votes)
print(v_accuracy)

In [ ]:
side_results = get_rulebook_results(rb_correct_side_alt, trajectories_dict)
side_correct, side_equal, side_incomparable, side_total, side_accuracy, side_weighted_accuracy, side_reasons, side_predictions = evaluate_rulebook_with_result_dict(rb_correct_side_alt, side_results, X, y, votes)
print(side_accuracy)

/opt/anaconda3/envs/benchmark/lib/python3.8/site-packages/shapely/measurement.py:72: RuntimeWarning: overflow encountered in distance
  return lib.distance(a, b, **kwargs)
/opt/anaconda3/envs/benchmark/lib/python3.8/site-packages/shapely/measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)


TypeError: '<' not supported between instances of 'Result' and 'Result'